# OCR RX 580 + CPU — versão revisada V2

Este notebook implementa OCR local robusto para PDFs e imagens. A rota padrão é **PaddleOCR 3.x + ONNX Runtime em CPU**; a RX 580 não acelera esse OCR. Use a GPU somente para VLM via Vulkan como revisão separada, nunca como substituição automática do texto extraído.

## Instalação

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -U jupyter paddleocr onnxruntime pymupdf opencv-python-headless numpy requests
jupyter notebook
```

Caso o backend `engine="onnxruntime"` da versão instalada do PaddleOCR falhe, instale `rapidocr` e altere `backend="rapidocr"` em `ConfigV2`.


In [ ]:
# OCR robusto V2 — execute esta célula após instalar dependências
# pip install -U paddleocr onnxruntime pymupdf opencv-python-headless numpy requests
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, time, math, hashlib, tempfile, csv, logging
import numpy as np
import cv2
import fitz

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
LOG = logging.getLogger("ocr_v2")

@dataclass(frozen=True)
class ConfigV2:
    entrada: str = "documentos"
    saida: str = "saida_ocr_v2"
    backend: str = "paddle"  # paddle (preferido) | rapidocr
    idioma: str = "pt"
    dpi: int = 300
    max_pixels_pagina: int = 18_000_000
    confianca_minima: float = 0.45
    pre_processar: bool = True
    deskew: bool = True
    denoise: bool = False
    cache: bool = True
    gerar_pdf_pesquisavel: bool = True
    extensoes: tuple = (".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")

CFG2 = ConfigV2()
IN2, OUT2 = Path(CFG2.entrada), Path(CFG2.saida)
IN2.mkdir(parents=True, exist_ok=True); OUT2.mkdir(parents=True, exist_ok=True)

def _id(path): return f"{path.stem}_{hashlib.sha256(str(path.resolve()).encode()).hexdigest()[:12]}"
def _atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False) as f:
        json.dump(obj, f, ensure_ascii=False, indent=2); tmp=Path(f.name)
    tmp.replace(path)
def _without_images(r): return {**r,"paginas":[{k:v for k,v in p.items() if k != "imagem"} for p in r["paginas"]]}
def _load_image(path):
    img=cv2.imdecode(np.fromfile(str(path),np.uint8),cv2.IMREAD_COLOR)
    if img is None: raise ValueError(f"Imagem inválida: {path.name}")
    if max(img.shape[:2]) > 16000: raise ValueError("Imagem recusada: lado acima de 16000 pixels")
    return img
def _pdf_pages(path,cfg):
    doc=fitz.open(str(path))
    try:
        if doc.needs_pass: raise ValueError("PDF protegido por senha")
        for n,page in enumerate(doc,1):
            scale=cfg.dpi/72
            pixels=page.rect.width*page.rect.height*scale*scale
            if pixels > cfg.max_pixels_pagina: scale*=math.sqrt(cfg.max_pixels_pagina/pixels)
            pix=page.get_pixmap(matrix=fitz.Matrix(scale,scale),alpha=False)
            raw=np.frombuffer(pix.samples,np.uint8).reshape(pix.height,pix.width,pix.n)
            image=cv2.cvtColor(raw,cv2.COLOR_RGB2BGR) if pix.n==3 else cv2.cvtColor(raw,cv2.COLOR_RGBA2BGR)
            yield n,image,round(scale*72,1)
    finally: doc.close()
def _pre(img,cfg):
    if not cfg.pre_processar:return img
    out=img
    if cfg.deskew:
        g=cv2.cvtColor(out,cv2.COLOR_BGR2GRAY); b=cv2.threshold(g,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]
        pts=np.column_stack(np.where(b>0))
        if len(pts)>=300:
            angle=cv2.minAreaRect(pts)[-1]; angle=-(90+angle) if angle < -45 else -angle
            if .35 <= abs(angle) <= 12:
                h,w=out.shape[:2]; out=cv2.warpAffine(out,cv2.getRotationMatrix2D((w/2,h/2),angle,1),(w,h),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_REPLICATE)
    if max(out.shape[:2]) < 1400:
        s=1400/max(out.shape[:2]); out=cv2.resize(out,None,fx=s,fy=s,interpolation=cv2.INTER_CUBIC)
    if cfg.denoise:out=cv2.fastNlMeansDenoisingColored(out,None,4,4,7,21)
    return out
def _order(items):
    if not items:return items
    hs=[]
    for x in items:
        b=np.asarray(x["box"],float); x["__x"]=float(b[:,0].mean()); x["__y"]=float(b[:,1].mean()); x["__h"]=float(max(1,np.ptp(b[:,1]))); hs.append(x["__h"])
    h=max(8,float(np.median(hs))); items.sort(key=lambda x:(round(x["__y"]/h),x["__x"]))
    for x in items: x.pop("__x");x.pop("__y");x.pop("__h")
    return items

class MotorOCRV2:
    def __init__(self,cfg):
        self.cfg=cfg
        if cfg.backend=="paddle":
            from paddleocr import PaddleOCR
            self.engine=PaddleOCR(lang=cfg.idioma,engine="onnxruntime",use_doc_orientation_classify=False,use_doc_unwarping=False,use_textline_orientation=True)
        elif cfg.backend=="rapidocr":
            from rapidocr import RapidOCR
            self.engine=RapidOCR()
        else:raise ValueError("backend: paddle ou rapidocr")
    def inferir(self,img):
        if self.cfg.backend=="rapidocr":
            raw=self.engine(img); raw=getattr(raw,"results",raw); raw=raw[0] if isinstance(raw,tuple) else raw
            out=[{"box":np.asarray(x[0],float).reshape(-1,2).tolist(),"texto":str(x[1]),"score":float(x[2])} for x in (raw or [])]
        else:
            res=list(self.engine.predict(img))
            if not res:return []
            d=res[0].json if hasattr(res[0],"json") else res[0]; d=d() if callable(d) else d
            fields=("rec_texts","rec_scores","rec_polys")
            if not isinstance(d,dict) or any(k not in d for k in fields):raise RuntimeError(f"Schema PaddleOCR inesperado: {list(d) if isinstance(d,dict) else type(d)}")
            out=[{"box":np.asarray(b,float).reshape(-1,2).tolist(),"texto":str(t),"score":float(s)} for t,s,b in zip(d["rec_texts"],d["rec_scores"],d["rec_polys"])]
        return _order(out)
def _signature(path,cfg):
    st=path.stat(); d=asdict(cfg);d.pop("entrada");d.pop("saida")
    return hashlib.sha256(json.dumps([str(path.resolve()),st.st_size,st.st_mtime_ns,d],sort_keys=True,default=list).encode()).hexdigest()
def processar_v2(path,cfg,motor):
    path=Path(path); base=OUT2/_id(path); cache=base.with_suffix(".cache.json"); sig=_signature(path,cfg)
    if cfg.cache and cache.exists():
        old=json.loads(cache.read_text(encoding="utf-8"))
        if old.get("assinatura")==sig:return old["resultado"]
    source=_pdf_pages(path,cfg) if path.suffix.lower()==".pdf" else [(1,_load_image(path),None)]
    pages=[]; errors=[]; started=time.perf_counter()
    for n,img,dpi_real in source:
        try:
            used=_pre(img,cfg); items=[z for z in motor.inferir(used) if z["score"]>=cfg.confianca_minima]; scores=[z["score"] for z in items]
            pages.append({"pagina":n,"dpi_real":dpi_real,"largura_px":used.shape[1],"altura_px":used.shape[0],"itens":items,"texto_ocr":"\n".join(z["texto"] for z in items),"confianca_media":round(float(np.mean(scores)),4) if scores else 0.,"imagem":used})
        except Exception as e:
            LOG.exception("Falha em %s, página %d",path.name,n);errors.append({"pagina":n,"erro":f"{type(e).__name__}: {e}"})
    result={"arquivo_origem":str(path),"id":_id(path),"tempo_s":round(time.perf_counter()-started,3),"paginas":pages,"erros":errors}
    if cfg.cache:_atomic_json(cache,{"assinatura":sig,"resultado":_without_images(result)})
    return result
def exportar_v2(r,cfg):
    base=OUT2/r["id"];_atomic_json(base.with_suffix(".json"),_without_images(r))
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",newline="",dir=OUT2,delete=False) as f:
        w=csv.writer(f);w.writerow(["arquivo","pagina","texto","score","box"])
        for p in r["paginas"]:
            for z in p["itens"]:w.writerow([r["arquivo_origem"],p["pagina"],z["texto"],z["score"],json.dumps(z["box"])])
        tmp=Path(f.name)
    tmp.replace(base.with_suffix(".csv"))
    if not cfg.gerar_pdf_pesquisavel:return
    doc=fitz.open()
    try:
        for p in r["paginas"]:
            if "imagem" not in p:continue
            img=p["imagem"];h,w=img.shape[:2];page=doc.new_page(width=w,height=h);ok,b=cv2.imencode(".jpg",img,[cv2.IMWRITE_JPEG_QUALITY,88])
            if not ok:raise RuntimeError("Falha ao codificar JPEG")
            page.insert_image(page.rect,stream=b.tobytes())
            for z in p["itens"]:
                a=np.asarray(z["box"]);x0,y0=a.min(axis=0);x1,y1=a.max(axis=0);page.insert_textbox(fitz.Rect(x0,y0,x1,y1),z["texto"],fontsize=max(5,(y1-y0)*.75),render_mode=3,overlay=True)
        with tempfile.NamedTemporaryFile(suffix=".pdf",dir=OUT2,delete=False) as f:tmp=Path(f.name)
        doc.save(tmp,garbage=3,deflate=True);tmp.replace(base.with_name(base.name+"_pesquisavel.pdf"))
    finally:doc.close()
def executar_v2(cfg=CFG2):
    files=sorted(p for p in IN2.rglob("*") if p.is_file() and p.suffix.lower() in cfg.extensoes)
    if not files:print("Nenhum arquivo em",IN2.resolve());return []
    motor=MotorOCRV2(cfg);results=[]
    for path in files:
        try:
            r=processar_v2(path,cfg,motor);exportar_v2(r,cfg);results.append(r);print(f"OK {path.name}: {len(r['paginas'])} páginas, {r['tempo_s']:.2f}s, {len(r['erros'])} erros")
        except Exception:LOG.exception("Documento ignorado: %s",path)
    _atomic_json(OUT2/"resumo.json",{"documentos":len(results),"paginas":sum(len(x["paginas"]) for x in results),"erros":sum(len(x["erros"]) for x in results),"config":asdict(cfg)})
    return results

# Uso: coloque arquivos em ./documentos e execute:
# resultados_v2 = executar_v2(CFG2)


## Execução

1. Coloque PDFs e imagens em `./documentos`.
2. Rode a célula de implementação.
3. Rode `resultados_v2 = executar_v2(CFG2)`.
4. Cada documento gera JSON estruturado, CSV de caixas e PDF pesquisável em `./saida_ocr_v2`.

### Banca de 9 especialistas

| Área | Correção aplicada |
|---|---|
| Arquitetura | ID por hash evita colisão de nomes |
| PaddleOCR 3.x | API `predict()` e schema são verificados explicitamente |
| ONNX Runtime | Sem multiprocessing agressivo por padrão |
| PDF | Limite de pixels impede consumo excessivo de RAM |
| Layout | Ordem de leitura por linhas e colunas |
| Confiabilidade | Escrita atômica e falhas isoladas por página |
| Dados | Cache determinístico por arquivo + configuração |
| Segurança | Limite de dimensão e nenhum `shell=True` |
| VLM | Não substitui OCR automaticamente; preserva rastreabilidade |

### Limites

Valide uma amostra representativa antes do lote. O PDF pesquisável contém a mesma imagem pré-processada analisada pelo OCR; assim, as caixas e a camada invisível de texto usam o mesmo sistema de coordenadas. Para manuscrito e tabelas complexas, mantenha revisão humana ou um módulo separado de estruturação.
